# Exercise 1 — Last week's revenue

⏱️ 15 minutes hands-on, then we discuss as a group.

## The situation

Priya runs Marketing. This landed in your inbox at 08:40 on Monday:

> Can you get me total revenue from orders over €100, broken down by region, for last week?
> The board deck is Thursday.
>
> Also — heads up for next sprint — we want to look at what people clicked on before they
> bought. Nothing formal yet, just exploring.

Two places the data lives:

| | |
|---|---|
| **The file drop** | `data/output/ex1_raw/` — whatever upstream exports, landed as-is. Orders as CSV, clickstream as JSON. |
| **The BI team's `orders` table** | one clean typed table, reloaded nightly. The existing dashboards run on it. |

Start with the file drop. It has everything, including the clickstream Priya wants next.

In [ ]:
import duckdb
from northtrail import local_data

con = duckdb.connect()
RAW = local_data("ex1_raw")

con.sql(f"SELECT * FROM glob('{RAW}/*')").show()

In [ ]:
# What does one of these files actually look like?
print(open(f"{RAW}/orders_part1.csv").readline())
print(open(f"{RAW}/orders_part2.csv").readline())
print(open(f"{RAW}/orders_part3.csv").readline())

In [ ]:
# Attempt: answer Priya's question straight off the files.
query = f'''
    SELECT region, round(sum(amount), 2) AS revenue, count(*) AS orders
    FROM read_csv_auto('{RAW}/orders_part*.csv')
    WHERE amount > 100
    GROUP BY region
    ORDER BY region
'''

try:
    con.sql(query).show()
except Exception as e:
    print(f"{type(e).__name__}: {e}")

## 🔍 What just happened?

The query didn't run. Read the error — DuckDB is telling you which file it choked on, and
it even suggests a fix: `union_by_name=true`.

Before you take that suggestion: what do you think it will do with a file that is missing
a column, or that spells a column differently?

In [ ]:
# Attempt, take two: take DuckDB's suggestion.
query = f'''
    SELECT region, round(sum(amount), 2) AS revenue, count(*) AS orders
    FROM read_csv_auto('{RAW}/orders_part*.csv', union_by_name=true)
    WHERE amount > 100
    GROUP BY region
    ORDER BY region
'''
con.sql(query).show()

That ran, and the numbers look like money. You could send this to Priya right now.

Don't. There are **800 orders** in that folder. Check the result before you trust it.

In [ ]:
# How many rows came through, and how many of them are missing the fields we grouped and filtered on?
con.sql(f'''
    SELECT
        count(*)                                   AS rows_read,
        count(*) FILTER (WHERE amount IS NULL)     AS null_amount,
        count(*) FILTER (WHERE region IS NULL)     AS null_region
    FROM read_csv_auto('{RAW}/orders_part*.csv', union_by_name=true)
''').show()

## 🔍 What just happened?

All 800 rows were read. But 250 of them have no `amount` — so `WHERE amount > 100` quietly
dropped every one of them — and another 250 have no `region`, so they landed in a group
that isn't a region at all.

Nothing failed. No warning. The number you were about to send Priya is missing roughly a
quarter of last week's revenue, and it looks completely reasonable.

Hold that thought and try the other source.

## The BI team's table

The BI team loads `orders` every night, straight from the order system rather than from this
file drop. The table has a schema, declared up front, and the load fails if incoming data
doesn't match it.

In [ ]:
# This is the BI team's table definition, reproduced here so you can query it locally.
con.sql('''
    CREATE TABLE orders (
        order_id    VARCHAR,
        customer_id VARCHAR,
        region      VARCHAR,
        amount      DECIMAL(10,2),
        order_ts    TIMESTAMP
    )
''')
con.sql(f"INSERT INTO orders SELECT * FROM read_csv_auto('{local_data('ex1_warehouse', 'orders.csv')}')")

con.sql('''
    SELECT region, round(sum(amount), 2) AS revenue, count(*) AS orders
    FROM orders
    WHERE amount > 100
    GROUP BY region
    ORDER BY region
''').show()

In [ ]:
# Same question, both sources, one number each.
lake = con.sql(f'''
    SELECT count(*) AS orders, round(sum(amount), 2) AS revenue
    FROM read_csv_auto('{RAW}/orders_part*.csv', union_by_name=true)
    WHERE amount > 100
''').fetchone()

warehouse = con.sql('''
    SELECT count(*) AS orders, round(sum(amount), 2) AS revenue
    FROM orders WHERE amount > 100
''').fetchone()

print(f"from the file drop : {lake[0]:>3} orders   EUR {lake[1]:>10,.2f}")
print(f"from the BI table  : {warehouse[0]:>3} orders   EUR {warehouse[1]:>10,.2f}")
# float() because the BI table types amount as DECIMAL and the CSVs infer DOUBLE.
gap_orders = warehouse[0] - lake[0]
gap_revenue = float(warehouse[1]) - float(lake[1])
print(f"                     {gap_orders:>3} orders   EUR {gap_revenue:>10,.2f} missing"
      f"  ({100 * gap_revenue / float(warehouse[1]):.0f}%)")

## 🔍 What just happened?

302 orders, €73,047.86. The file-drop answer was 222 orders and €53,879.45 — **€19,168 of
real revenue, 26% of the total, silently gone**.

That settles Priya's first question. Now her second one: what were people clicking on?

In [ ]:
# Does the BI team's system have anything to answer that with?
con.sql("SHOW TABLES").show()

# And the file drop?
con.sql(f'''
    SELECT event_type, count(*) AS n
    FROM read_json_auto('{RAW}/events.json')
    GROUP BY event_type
    ORDER BY n DESC
''').show()

## 🔍 What just happened?

The clean system has exactly one table — orders — and no clickstream at all. Getting one
means someone designs a schema, writes a load job, and ships it. Call it a week, and Priya
said "nothing formal yet, just exploring."

The file drop answered it in one query, on data with inconsistent keys and nested objects
that nobody has ever modelled.

So each source solved precisely the problem the other one couldn't.

## 💡 Concept: lake, warehouse, lakehouse

A **data lake** is a pile of files in cheap storage. Anything can land in it immediately, in
whatever shape it arrives — which is why the clickstream question took one query, and why
the revenue number was wrong without telling you. Nothing checks the files against a schema,
because there is no schema; you find out what's in a file when you read it.

A **data warehouse** is a system that only accepts data matching a schema it declares up
front, and stores it in its own internal format. That constraint is what made the revenue
number correct, and it is the same constraint that means the clickstream isn't there yet.

Neither is better. You are picking which problem you'd rather have. A **lakehouse** is the
attempt to keep the lake's cheap open files while adding the guarantees the warehouse got
from owning its storage — and the rest of today is about how that attempt actually works,
and where it leaks.

## Your turn

Five real situations. For each, write **lake**, **warehouse**, or **lakehouse**, and one
line on what decided it. Argue about the ones you disagree on — several are genuinely close.

| # | Situation | Your call | Because |
|---|---|---|---|
| 1 | Finance needs the quarterly revenue figure to be identical every time it's run, auditable, with five years of history. | | |
| 2 | A data scientist wants to dig through two years of raw app logs nobody has modelled, to see if there's a churn signal in there. | | |
| 3 | Six teams query the same hourly-refreshed sales data in SQL, and an ML job needs the same data from Python. | | |
| 4 | A 30-person startup, one data engineer, 50 GB of data, needs BI dashboards live next month. | | |
| 5 | Seven years of raw regulatory submissions must be retained cheaply. Queried maybe twice a year, when an auditor asks. | | |

## Debrief

- Priya's number was wrong by 26% and nothing anywhere reported an error. Which of the two
  systems would have caught it, and what exactly was doing the catching?
- The BI team spent a week modelling that `orders` table. Describe a situation where that
  week is obviously worth it, and one where spending it would be a mistake.
- The lakehouse pitch is "warehouse guarantees, on lake storage." What would you want to see
  before you believed it?